# 05. 日本・台湾の残差比較とエコノミスト論争への対応

03で行った年別横断面回帰・年固定効果付きプール回帰の結果を踏まえ、
時系列の変化を検討する。年効果なしプール回帰の
残差を2005年・2023年の2時点だけで比較すると単調な変化に見えるが、年別回帰では
両国とも残差が大きく上下しており、単調なトレンドは確認できない。

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUT_TABLES_DIR = BASE_DIR / "output" / "tables"
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

FOCUS_COUNTRIES = {"JPN": "日本", "KOR": "韓国", "TWN": "台湾"}
year_summary_df = pd.read_csv(OUTPUT_TABLES_DIR / "penn_regression_by_year_summary.csv")
print(f"年別回帰サマリー: {len(year_summary_df)}年分")

年別回帰サマリー: 19年分


## 1. 日本・台湾の残差の年別推移（年別横断面回帰）

03の年別横断面回帰の結果をそのまま示す。プール回帰の残差を2時点だけ比較するのではなく、
全19年の推移を見る。

In [2]:
print(year_summary_df[["year", "slope", "JPN_resid_pct", "KOR_resid_pct", "TWN_resid_pct"]].round(1).to_string(index=False))

jpn_series = year_summary_df["JPN_resid_pct"]
twn_series = year_summary_df["TWN_resid_pct"]
print()
print(f"日本の残差：最小{jpn_series.min():+.1f}%（{int(year_summary_df.loc[jpn_series.idxmin(), 'year'])}年）"
      f"〜最大{jpn_series.max():+.1f}%（{int(year_summary_df.loc[jpn_series.idxmax(), 'year'])}年）、"
      f"標準偏差{jpn_series.std():.1f}pt")
print(f"台湾の残差：最小{twn_series.min():+.1f}%（{int(year_summary_df.loc[twn_series.idxmin(), 'year'])}年）"
      f"〜最大{twn_series.max():+.1f}%（{int(year_summary_df.loc[twn_series.idxmax(), 'year'])}年）、"
      f"標準偏差{twn_series.std():.1f}pt")
print()
print("いずれも年によって大きく変動しており、単調な『収束』『乖離拡大』というトレンドは")
print("年別回帰からは確認できない。ただし台湾は全19年にわたって一貫して残差が大きく負で")
print("あり続けている点は安定した特徴である。")

 year  slope  JPN_resid_pct  KOR_resid_pct  TWN_resid_pct
 2005    0.6           11.5          -17.3          -54.0
 2006    0.7            5.7           -8.2          -54.1
 2007    0.7           -3.4          -12.5          -57.6
 2008    0.8            8.1          -23.8          -54.7
 2009    0.7           24.5          -31.1          -55.7
 2010    0.7           28.1          -24.6          -56.1
 2011    0.8           37.4          -19.6          -54.4
 2012    0.9           45.6          -12.4          -54.0
 2013    0.9           10.6           -4.4          -54.8
 2014    0.8            7.3           -2.2          -55.8
 2015    1.0           10.6            1.9          -51.9
 2016    1.1           34.5           -0.5          -50.1
 2017    1.0           29.8            1.6          -46.2
 2018    1.0           30.2           -2.1          -45.9
 2019    1.0           48.9            1.6          -44.0
 2020    1.0           48.4           -6.1          -47.5
 2021    0.9  

## 2. 2023年時点の中心的な結論（横断面）

03の高所得クラブ2023年単年の結果を再掲する。これが本稿で安全に主張できる
中心的な結論である。

In [3]:
hi_fit = pd.read_csv(PROCESSED_DIR / "penn_regression_2023_highincome.csv")
for cc, name in FOCUS_COUNTRIES.items():
    row = hi_fit[hi_fit["countrycode"] == cc].iloc[0]
    rank = int(hi_fit["resid"].rank().loc[hi_fit["countrycode"] == cc].iloc[0])
    print(f"{name}: 残差{row['resid_pct']:+.1f}%（40か国中{rank}位、残差が低いほど所得水準対比で物価水準が低い）")

日本: 残差+17.9%（40か国中32位、残差が低いほど所得水準対比で物価水準が低い）
韓国: 残差-11.2%（40か国中8位、残差が低いほど所得水準対比で物価水準が低い）
台湾: 残差-51.9%（40か国中1位、残差が低いほど所得水準対比で物価水準が低い）


## 3. エコノミスト論争への回答

『エコノミスト』誌のGDP調整済みビッグマック指数（2025年11月、-55%）と、
台湾中央銀行が引用した野村のiPhone指数（2016年算出、+17.1%）を、本稿のPenn回帰
（2023年、-51.9%）と比較する。両者は算出年・価格バスケット・所得指標が異なるため、
厳密な方法論の比較ではなく、**推計値の水準を並べる**にとどめる。

In [4]:
comparison = pd.DataFrame([
    {"指標": "エコノミスト GDP調整済みビッグマック指数（2025年11月）", "所得調整": "あり", "示す乖離": "台湾ドルは米ドルに対し55%割安"},
    {"指標": "野村 iPhone指数（2016年算出、台湾中銀が2025年に引用）", "所得調整": "なし", "示す乖離": "台湾ドルは米ドルに対し17.1%割高"},
    {"指標": "本論文：Penn回帰・高所得クラブ2023年単年", "所得調整": "あり（40か国の所得水準で回帰）", "示す乖離": "台湾の相対物価水準は回帰予測値を51.9%下回る"},
])
print(comparison.to_string(index=False))
print()
print("エコノミスト誌の値（2025年、-55%）と本論文の値（2023年、-51.9%）は数値上近いが、")
print("対象年・データソース・回帰標本が異なるため、これを『所得調整という方法論が同じだから")
print("結論が収束する』と解釈することは適切でない。推計値が近い、という事実の指摘にとどめる。")

                                指標             所得調整                     示す乖離
  エコノミスト GDP調整済みビッグマック指数（2025年11月）               あり         台湾ドルは米ドルに対し55%割安
野村 iPhone指数（2016年算出、台湾中銀が2025年に引用）               なし       台湾ドルは米ドルに対し17.1%割高
          本論文：Penn回帰・高所得クラブ2023年単年 あり（40か国の所得水準で回帰） 台湾の相対物価水準は回帰予測値を51.9%下回る

エコノミスト誌の値（2025年、-55%）と本論文の値（2023年、-51.9%）は数値上近いが、
対象年・データソース・回帰標本が異なるため、これを『所得調整という方法論が同じだから
結論が収束する』と解釈することは適切でない。推計値が近い、という事実の指摘にとどめる。


## 4. 結果の保存

In [5]:
out_path = OUTPUT_TABLES_DIR / "paper2_final_summary_v2.csv"
year_summary_df.to_csv(out_path, index=False)
print(f"保存先: {(out_path).relative_to(BASE_DIR)}")

保存先: output\tables\paper2_final_summary_v2.csv


## まとめと限界

- **安全に主張できる中心結論**：本稿の標本・仕様（2023年、高所得クラブ40か国）では、
  台湾の残差が最小であり、日本の残差は正（所得水準から予想される水準を上回る）
  であった。
- **撤回した主張**：「日本は収束・台湾は乖離拡大」という時系列のトレンド解釈は、
  年効果なしプール回帰の2時点比較に基づくものであり、年別回帰では両国とも
  大きな年次変動があるため撤回する。台湾が全期間にわたって一貫して残差が大きく
  負であり続けている点（水準の安定性）は残せるが、単調なトレンドの主張はしない。
- **エコノミスト論争**：所得調整済みの推計値同士が数値上近いことは指摘できるが、
  対象年・データが異なることに加え、測っている概念自体（通貨の割高・割安 対
  所得条件付き物価水準の乖離）も異なるため、方法論の一致による『結論の収束』
  とまでは主張しない。
- 除外国リストへの感度（03§8）により、台湾の残差の具体的な大きさ（-42%〜-52%）は
  リストの選び方に依存する。残差の符号（予測値を下回る方向）と相対的な大きさ
  （40か国中最小の部類）は維持される。